![image_1781377328251.png](./image_1781377328251.png "image_1781377328251.png")

In [0]:
%sql
CREATE OR REPLACE TABLE car_insurance_demo.03_gold.customer_claim_policy_telematics_predicted AS 
  SELECT
    t.*,
    a.* EXCEPT (chassis_no, claim_no)
  FROM
    car_insurance_demo.03_gold.customer_claim_policy_telematics t
    JOIN car_insurance_demo.03_gold.claim_images_predicted a USING(claim_no)
     

num_affected_rows,num_inserted_rows


In [0]:

%sql
CREATE OR REPLACE TABLE car_insurance_demo.02_silver.claims_rules (
  rule_id BIGINT GENERATED ALWAYS AS IDENTITY,
  rule STRING, 
  check_name STRING,
  check_code STRING,
  check_severity STRING,
  is_active Boolean
);

In [0]:
def insert_rule(rule, name, code, severity, is_active):
  spark.sql(f"INSERT INTO car_insurance_demo.02_silver.claims_rules(rule,check_name, check_code, check_severity,is_active) values('{rule}', '{name}', '{code}', '{severity}', {is_active})")


In [0]:
invalid_policy_date = '''
  CASE WHEN to_date(pol_eff_date, "yyyy-MM-dd") < to_date(claim_date) and to_date(pol_expiry_date, "yyyy-MM-dd") < to_date(claim_date) THEN "VALID" 
  ELSE "NOT VALID"  
  END
'''

insert_rule('invalid policy date', 'valid_date', invalid_policy_date, 'HIGH', True)

In [0]:
exceeds_policy_amount = '''
CASE WHEN  sum_insured >= total 
    THEN "claim value in the range of premium"
    ELSE "claim value more than premium"
END 
'''
insert_rule('exceeds policy amount', 'valid_amount', exceeds_policy_amount, 'HIGH', True)

In [0]:
severity_mismatch = '''
CASE WHEN    severity="Total Loss" AND damage_prediction.label = "major" THEN  "Severity matches the report"
       WHEN  severity="Major Damage" AND damage_prediction.label = "minor"  THEN  "Severity matches the report"
       WHEN  (severity="Minor Damage" OR severity="Trivial Damage") AND damage_prediction.label = "ok" THEN  "Severity matches the report"
       ELSE "Severity does not match"
END 
'''

insert_rule('severity mismatch', 'reported_severity_check', severity_mismatch, 'HIGH', True)

In [0]:
exceeds_speed = '''
CASE WHEN  telematics_speed <= 45 and telematics_speed > 0 THEN  "Normal Speed"
       WHEN telematics_speed > 45 THEN  "High Speed"
       ELSE "Invalid speed"
END
'''
insert_rule('exceeds speed', 'speed_check', exceeds_speed, 'HIGH', True)

In [0]:
release_funds = '''
CASE WHEN  reported_severity_check="Severity matches the report" and valid_amount="claim value in the range of premium" and valid_date="VALID" then "release funds"
       ELSE "claim needs more investigation" 
END
'''
insert_rule('release funds', 'fund_release', release_funds, 'HIGH', True)

In [0]:
from pyspark.sql.functions import expr

df = spark.sql("SELECT * FROM car_insurance_demo.03_gold.customer_claim_policy_telematics_predicted")
rules = spark.sql('SELECT * FROM car_insurance_demo.02_silver.claims_rules where is_active=true order by rule_id').collect()
for rule in rules:
  print(rule.rule, rule.check_code)
  df=df.withColumn(rule.check_name, expr(rule.check_code))

#overwrite table with new insights
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("car_insurance_demo.03_gold.claim_insights")

invalid policy date 
  CASE WHEN to_date(pol_eff_date, "yyyy-MM-dd") < to_date(claim_date) and to_date(pol_expiry_date, "yyyy-MM-dd") < to_date(claim_date) THEN "VALID" 
  ELSE "NOT VALID"  
  END

exceeds policy amount 
CASE WHEN  sum_insured >= total 
    THEN "claim value in the range of premium"
    ELSE "claim value more than premium"
END 

severity mismatch 
CASE WHEN    severity="Total Loss" AND damage_prediction.label = "major" THEN  "Severity matches the report"
       WHEN  severity="Major Damage" AND damage_prediction.label = "minor"  THEN  "Severity matches the report"
       WHEN  (severity="Minor Damage" OR severity="Trivial Damage") AND damage_prediction.label = "ok" THEN  "Severity matches the report"
       ELSE "Severity does not match"
END 

exceeds speed 
CASE WHEN  telematics_speed <= 45 and telematics_speed > 0 THEN  "Normal Speed"
       WHEN telematics_speed > 45 THEN  "High Speed"
       ELSE "Invalid speed"
END

release funds 
CASE WHEN  reported_severity_chec

In [0]:
display(rules)

rule_id,rule,check_name,check_code,check_severity,is_active
1,invalid policy date,valid_date,"CASE WHEN to_date(pol_eff_date, ""yyyy-MM-dd"") < to_date(claim_date) and to_date(pol_expiry_date, ""yyyy-MM-dd"") < to_date(claim_date) THEN ""VALID"" ELSE ""NOT VALID"" END",HIGH,true
2,exceeds policy amount,valid_amount,"CASE WHEN sum_insured >= total THEN ""claim value in the range of premium"" ELSE ""claim value more than premium"" END",HIGH,true
3,severity mismatch,reported_severity_check,"CASE WHEN severity=""Total Loss"" AND damage_prediction.label = ""major"" THEN ""Severity matches the report"" WHEN severity=""Major Damage"" AND damage_prediction.label = ""minor"" THEN ""Severity matches the report"" WHEN (severity=""Minor Damage"" OR severity=""Trivial Damage"") AND damage_prediction.label = ""ok"" THEN ""Severity matches the report"" ELSE ""Severity does not match"" END",HIGH,true
4,exceeds speed,speed_check,"CASE WHEN telematics_speed <= 45 and telematics_speed > 0 THEN ""Normal Speed"" WHEN telematics_speed > 45 THEN ""High Speed"" ELSE ""Invalid speed"" END",HIGH,true
5,release funds,fund_release,"CASE WHEN reported_severity_check=""Severity matches the report"" and valid_amount=""claim value in the range of premium"" and valid_date=""VALID"" then ""release funds"" ELSE ""claim needs more investigation"" END",HIGH,true


In [0]:
%sql
SELECT *  FROM car_insurance_demo.03_gold.claim_insights
LIMIT 1

claim_no,CHASSIS_NO,policy_no,claim_date,months_as_customer,injury,property,vehicle,total,collision_type,number_of_vehicles_involved,age,insured_relationship,license_issue_date,date,hour,type,severity,number_of_witnesses,suspicious_activity,CUST_ID,POLICYTYPE,POL_ISSUE_DATE,POL_EFF_DATE,POL_EXPIRY_DATE,MAKE,MODEL,MODEL_YEAR,USE_OF_VEHICLE,PRODUCT,SUM_INSURED,premium,DEDUCTABLE,customer_id,date_of_birth,borough,neighborhood,zip_code,name,firstname,lastname,address,lat_long,telematics_speed,telematics_latitude,telematics_longitude,image_name,path,modificationTime,length,content,damage_prediction,image_id,_rescued_data,valid_date,valid_amount,reported_severity_check,speed_check,fund_release
f86f9ba4-a561-4b78-8670-4eb5660f38d5,LSJA36E30LZ021981,102153344,2021-01-02,59,520,520,3640,4680,Rear Collision,4,25,own-child,2011-02-09,2020-08-04,0,Multi-vehicle Collision,Total Loss,2,false,12714.0,COMP,2020-12-23,2020-12-23,2022-01-22,MG,MG5,2020.0,PRIVATE,M 2019,34020.0,1490.0,1000,12714,2009-06-13,BROOKLYN,BOROUGH PARK,11219,"Horn, Lara",Lara,Horn,"BROOKLYN, 11219","List(82.044136, 130.77641)",31.59356239810586,40.78434902499998,-73.90740773333337,1_High.jpg,dbfs:/Volumes/car_insurance_demo/01_bronze/data_files/images/1_High.jpg,2026-06-10T16:51:44.000Z,634689,...,"List(ok, 0.4007367491722107)",1,null,NOT VALID,claim value in the range of premium,Severity does not match,Normal Speed,claim needs more investigation
